# Marmousi Elastic OPT Debug

Small `2DsismoTimeIsoHetero` check using Marmousi-derived `rho`, `vp`, and `vs`. The material variables passed to the equation are `rho`, `lambda`, and `mu` in MKS units.


## 1. Setup


In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\n# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")
@show Metal.devices()

using JLD2

include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints

include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .flexOPT

include("temporaryHelpers.jl")\n

## 2. Homogeneous Elastic Smoke Test

This first checks the LHS propagation with a Gaussian initial field. The CFL is intentionally conservative because the current OPT3 elastic stencil can be unstable; if it still blows up, inspect the LHS stencil before testing source `Γ`.


In [ ]:
shape_homo = (81, 81)
dx_homo = 1.0e3
rho0_homo = 2500.0
vp0_homo = 3200.0
vs0_homo = 1800.0

rho_homo = fill(rho0_homo, shape_homo)
vp_homo = fill(vp0_homo, shape_homo)
vs_homo = fill(vs0_homo, shape_homo)
lambda_homo = rho_homo .* vp_homo.^2 .- 2 .* rho_homo .* vs_homo.^2
mu_homo = rho_homo .* vs_homo.^2
models_homo_mks = [rho_homo, lambda_homo, mu_homo]

# Use a longer visible run than the tiny smoke test. With cfl=0.08 and Nt=30,
# the P wave travels only about 1-2 grid cells, so P/S are almost invisible.
cfl_homo = 0.20
cfl_info_homo = cfl_diagnostics(vp_homo, (dx_homo, dx_homo, 1.0); cfl_safety=cfl_homo)
dt_homo = cfl_info_homo.suggested_dt_2D
delta_homo_mks = (dx_homo, dx_homo, dt_homo)

# Dimensionless elastic recipe: rho'=1, lambda'=(lambda/rho)*(dt/dx)^2, mu'=(mu/rho)*(dt/dx)^2.
# This is the elastic analogue of acoustic velocity_cfl = v*dt/dx.
rho_homo_nd = ones(Float64, shape_homo)
lambda_homo_nd = (lambda_homo ./ rho_homo) .* (dt_homo / dx_homo)^2
mu_homo_nd = (mu_homo ./ rho_homo) .* (dt_homo / dx_homo)^2
models_homo = [rho_homo_nd, lambda_homo_nd, mu_homo_nd]
delta_homo = (1.0, 1.0, 1.0)

sourcePoint_homo = CartesianIndex(cld(shape_homo[1], 2), cld(shape_homo[2], 2))
Nt_homo = 120
store_every_homo = 5
total_time_homo = Nt_homo * dt_homo
source_f0_homo = min(max(cfl_info_homo.suggested_f0, 0.15), 1 / (8dt_homo))
source_t0_homo = min(12dt_homo, 0.25total_time_homo)
source_amplitude_homo = 1.0

@show delta_homo_mks delta_homo extrema(lambda_homo_nd) extrema(mu_homo_nd)
@show sourcePoint_homo Nt_homo store_every_homo total_time_homo source_f0_homo source_t0_homo
@show vp0_homo * total_time_homo / dx_homo vs0_homo * total_time_homo / dx_homo


In [ ]:
elasticOPT_homo = build_opt_prepared(
    "2DsismoTimeIsoHeteroForce",
    models_homo,
    delta_homo;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="homogeneous_elastic_OPT_force",
)
preparedElastic_homo = elasticOPT_homo.prepared
preparedFD3_homo = prepare_fd2d_elastic_pointwise_baseline(
    rho_homo_nd,
    lambda_homo_nd,
    mu_homo_nd,
    delta_homo;
    spatial_order=3,
    source_scale=1.0,
)

elastic_A_report_homo = implicit_matrix_report(preparedElastic_homo)
fd3_A_report_homo = implicit_matrix_report(preparedFD3_homo)

@show preparedElastic_homo.spaceShape preparedElastic_homo.NField preparedElastic_homo.NForceField
@show elastic_A_report_homo
@show fd3_A_report_homo

rho0_nd_homo = rho_homo_nd[sourcePoint_homo]
lambda0_nd_homo = lambda_homo_nd[sourcePoint_homo]
mu0_nd_homo = mu_homo_nd[sourcePoint_homo]
lhs_diag_homo = elastic_lhs_local_diagnostics(elasticOPT_homo.numOps, sourcePoint_homo, rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo)
fd3_diag_homo = prepared_elastic_lhs_local_diagnostics(preparedFD3_homo, sourcePoint_homo, rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo)
@show rho0_nd_homo lambda0_nd_homo mu0_nd_homo
@show lhs_diag_homo.rows[1]
@show lhs_diag_homo.rows[2]
@show fd3_diag_homo.rows[1]
@show fd3_diag_homo.rows[2]
lhs_diag_homo.matrices


In [ ]:
recipe_lhs_symbolic_matrices(elasticOPT_homo.optRec; iExpr=1, iField=1)

In [ ]:
using Symbolics

In [ ]:
semi_11 = recipe_lhs_evaluated_summary(
    elasticOPT_homo.optRec,
    (rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo);
    iExpr=1,
    iField=1,
)

semi_12 = recipe_lhs_evaluated_summary(
    elasticOPT_homo.optRec,
    (rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo);
    iExpr=1,
    iField=2,
)

semi_11

In [ ]:
function symbol_at_pi(summary)
    Dict(row.time_role => sum(((-1)^(i+j)) * row.matrix[i,j]
                              for i in 1:3, j in 1:3)
         for row in summary)
end

symbol_at_pi(semi_11)
symbol_at_pi(semi_12)

In [ ]:
@show semi_11
@show lhs_diag_homo.matrices[(1,1)]

In [ ]:
recette_homo = elasticOPT_homo.optRec["recette"]
Asemi = recette_homo.lhs.Ajiννᶜ
size(Asemi)

In [ ]:
Asemi

In [ ]:
lhs_diag_homo.rows[1].comparison
lhs_diag_homo.rows[2].comparison
lhs_diag_homo.rows[3].comparison
lhs_diag_homo.rows[4].comparison

In [ ]:
lhs_diag_homo.rows[1].worst_stability
lhs_diag_homo.rows[2].worst_stability
lhs_diag_homo.rows[3].worst_stability
lhs_diag_homo.rows[4].worst_stability

In [ ]:
for key in keys(lhs_diag_homo.matrices)
    println("\nBLOCK ", key)
    for m in lhs_diag_homo.matrices[key]
        println(m.time_role)
        display(m.matrix)
    end
end

In [ ]:
@show lhs_diag_homo.matrices[(1,1)]
@show lhs_diag_homo.matrices[(1,2)]
@show lhs_diag_homo.comparisons[(1,1)]
@show lhs_diag_homo.comparisons[(1,2)]

In [ ]:
lhs_diag_homo.comparisons[(1,1)]
lhs_diag_homo.matrices[(1,1)]
lhs_diag_homo.stability[(1,1)][1]

In [ ]:
initial_ux_homo = gaussian_field(shape_homo, sourcePoint_homo; sigma=8.0, amplitude=1.0)
initial_homo = zeros(Float64, shape_homo..., 2)
initial_homo[:, :, 1] .= initial_ux_homo

frames_gauss_homo_full = propagate_linear_frames_with_source(
    preparedElastic_homo,
    Nt_homo;
    initialPast=initial_homo,
    initialPresent=initial_homo,
    store_every=store_every_homo,
    blowup_limit=1e6,
)
frames_gauss_fd3_homo_full = propagate_linear_frames_with_source(
    preparedFD3_homo,
    Nt_homo;
    initialPast=initial_homo,
    initialPresent=initial_homo,
    store_every=store_every_homo,
    blowup_limit=1e6,
)
ux_gauss_homo = component_frames(frames_gauss_homo_full, 1)
uz_gauss_homo = component_frames(frames_gauss_homo_full, 2)
ux_gauss_fd3_homo = component_frames(frames_gauss_fd3_homo_full, 1)
uz_gauss_fd3_homo = component_frames(frames_gauss_fd3_homo_full, 2)
@show wavefield_snapshot_report(ux_gauss_homo)[end] wavefield_snapshot_report(uz_gauss_homo)[end]
@show wavefield_snapshot_report(ux_gauss_fd3_homo)[end] wavefield_snapshot_report(uz_gauss_fd3_homo)[end]
@show frame_difference_report(uz_gauss_homo, uz_gauss_fd3_homo)[end]

# Direct vertical body-force test: fx=0, fz=1. This should radiate both P and S components.
timeSignal_homo = source_time_samples(Nt_homo, dt_homo, preparedElastic_homo.timePointsUsedForOneStep; t0=source_t0_homo, f0=source_f0_homo)
sourceFull_fz_homo = direct_force_source_full(preparedElastic_homo, sourcePoint_homo, timeSignal_homo; fx=0.0, fz=1.0, amplitude=source_amplitude_homo)
sourceFull_fz_fd3_homo = direct_force_source_full(preparedFD3_homo, sourcePoint_homo, timeSignal_homo; fx=0.0, fz=1.0, amplitude=source_amplitude_homo)
source_peak_index_homo = argmax(abs.(timeSignal_homo))
source_peak_time_homo = (source_peak_index_homo - 1) * dt_homo
source_rhs_homo = source_rhs_diagnostics(preparedElastic_homo, sourceFull_fz_homo; it=min(source_peak_index_homo, Nt_homo))
@show maximum(abs, timeSignal_homo) source_peak_index_homo source_peak_time_homo maximum(abs, sourceFull_fz_homo)
@show source_rhs_homo.b_max source_rhs_homo.b_norm source_rhs_homo.b_argmax

frames_force_homo_full = propagate_linear_frames_with_source(
    preparedElastic_homo,
    Nt_homo;
    sourceFull=sourceFull_fz_homo,
    store_every=store_every_homo,
    blowup_limit=1e6,
)
frames_force_fd3_homo_full = propagate_linear_frames_with_source(
    preparedFD3_homo,
    Nt_homo;
    sourceFull=sourceFull_fz_fd3_homo,
    store_every=store_every_homo,
    blowup_limit=1e6,
)
ux_frames_homo = component_frames(frames_force_homo_full, 1)
uz_frames_homo = component_frames(frames_force_homo_full, 2)
ux_frames_fd3_homo = component_frames(frames_force_fd3_homo_full, 1)
uz_frames_fd3_homo = component_frames(frames_force_fd3_homo_full, 2)

ux_report_homo = wavefield_snapshot_report(ux_frames_homo)
uz_report_homo = wavefield_snapshot_report(uz_frames_homo)
ux_report_fd3_homo = wavefield_snapshot_report(ux_frames_fd3_homo)
uz_report_fd3_homo = wavefield_snapshot_report(uz_frames_fd3_homo)
@show length(frames_force_homo_full) ux_report_homo[1] ux_report_homo[end]
@show uz_report_homo[1] uz_report_homo[end]
@show uz_report_fd3_homo[1] uz_report_fd3_homo[end]
@show frame_difference_report(uz_frames_homo, uz_frames_fd3_homo)[end]


In [ ]:
display(plot_wave_snapshots(ux_gauss_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT Gaussian ux"))
display(plot_wave_snapshots(uz_gauss_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT Gaussian uz"))
display(plot_wave_snapshots(ux_gauss_fd3_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic FD3 Gaussian ux"))
display(plot_wave_snapshots(uz_gauss_fd3_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic FD3 Gaussian uz"))


In [ ]:
display(plot_wave_snapshots(ux_frames_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT fz source ux"))
display(plot_wave_snapshots(uz_frames_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT fz source uz"))
display(plot_wave_snapshots(ux_frames_fd3_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic FD3 fz source ux"))
display(plot_wave_snapshots(uz_frames_fd3_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic FD3 fz source uz"))


## 3. Reduced Elastic Marmousi Material


In [ ]:
marmousi = load(joinpath(@__DIR__, "tmp/seismicModelMarmousi.jld2"), "output")

shape = (41, 41)
downsample_step = 8
rho_raw = downsample_center_crop(marmousi.ρ, shape; step=downsample_step)
vp_raw = downsample_center_crop(marmousi.Vpv, shape; step=downsample_step)
vs_raw = downsample_center_crop(marmousi.Vsv, shape; step=downsample_step)

rho, lambda, mu, vp, vs = elastic_lame_from_rho_vp_vs(rho_raw, vp_raw, vs_raw)

# First small linear test: avoid a singular pure-fluid block. Set to 0.0 when testing the free-surface/fluid behavior explicitly.
vs_floor = 500.0
if vs_floor > 0
    vs = max.(vs, vs_floor)
    mu = rho .* vs.^2
    lambda = rho .* vp.^2 .- 2 .* mu
end

dx = 1.0e3 * downsample_step
cfl = 0.20
cfl_info = cfl_diagnostics(vp, (dx, dx, 1.0); cfl_safety=cfl)
dt = cfl_info.suggested_dt_2D
delta_mks = (dx, dx, dt)

# Dimensionless elastic recipe for visible/stable debug propagation.
rho_nd = ones(Float64, shape)
lambda_nd = (lambda ./ rho) .* (dt / dx)^2
mu_nd = (mu ./ rho) .* (dt / dx)^2
models_elastic = [rho_nd, lambda_nd, mu_nd]
delta = (1.0, 1.0, 1.0)

sourcePoint = CartesianIndex(cld(size(rho, 1), 2), cld(size(rho, 2), 2))
Nt = 160
store_every = 5
total_time = Nt * dt

# Keep the source peak inside this debug run, but high enough to show Marmousi wavefront distortion.
source_f0 = min(max(cfl_info.suggested_f0, 0.04), 1 / (8dt))
source_t0 = min(12dt, 0.25total_time)
source_amplitude = 1.0

@show size(rho) extrema(rho) extrema(vp) extrema(vs) extrema(lambda) extrema(mu)
@show delta_mks delta extrema(lambda_nd) extrema(mu_nd)
@show sourcePoint Nt store_every total_time source_f0 source_t0
@show maximum(vp) * total_time / dx maximum(vs) * total_time / dx


## 4. Build 2DsismoTimeIsoHetero OPT


In [ ]:
elasticOPT = build_opt_prepared(
    "2DsismoTimeIsoHetero",
    models_elastic,
    delta;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="Marmousi_elastic_moment_OPT",
)
preparedElastic = elasticOPT.prepared

elasticOPT_force = build_opt_prepared(
    "2DsismoTimeIsoHeteroForce",
    models_elastic,
    delta;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="Marmousi_elastic_force_OPT",
)
preparedElastic_force = elasticOPT_force.prepared
preparedFD3 = prepare_fd2d_elastic_pointwise_baseline(
    rho_nd,
    lambda_nd,
    mu_nd,
    delta;
    spatial_order=3,
    source_scale=1.0,
)

elastic_A_report = implicit_matrix_report(preparedElastic)
elastic_force_A_report = implicit_matrix_report(preparedElastic_force)
fd3_A_report = implicit_matrix_report(preparedFD3)

@show preparedElastic.spaceShape preparedElastic.NpointsSpace preparedElastic.NField preparedElastic.NForceField preparedElastic.timePointsUsedForOneStep
@show preparedElastic_force.NForceField preparedFD3.NForceField
@show size(preparedElastic.A_unknown) nnz(preparedElastic.A_unknown) nnz(preparedElastic.L_known) nnz(preparedElastic.R_force)
@show elastic_A_report
@show elastic_force_A_report
@show fd3_A_report

rho0_nd = rho_nd[sourcePoint]
lambda0_nd = lambda_nd[sourcePoint]
mu0_nd = mu_nd[sourcePoint]
lhs_diag_marm = elastic_lhs_local_diagnostics(elasticOPT.numOps, sourcePoint, rho0_nd, lambda0_nd, mu0_nd)
lhs_diag_force_marm = elastic_lhs_local_diagnostics(elasticOPT_force.numOps, sourcePoint, rho0_nd, lambda0_nd, mu0_nd)
fd3_diag_marm = prepared_elastic_lhs_local_diagnostics(preparedFD3, sourcePoint, rho0_nd, lambda0_nd, mu0_nd)
@show rho0_nd lambda0_nd mu0_nd
@show lhs_diag_force_marm.rows[1]
@show lhs_diag_force_marm.rows[2]
@show fd3_diag_marm.rows[1]
@show fd3_diag_marm.rows[2]
lhs_diag_force_marm.matrices


## 5. Marmousi Source And Linear Propagation


In [ ]:
initial_ux = gaussian_field(size(rho), sourcePoint; sigma=8.0, amplitude=1.0)
initial = zeros(Float64, size(rho)..., 2)
initial[:, :, 1] .= initial_ux

frames_gauss_full = propagate_linear_frames_with_source(
    preparedElastic_force,
    Nt;
    initialPast=initial,
    initialPresent=initial,
    store_every=store_every,
    blowup_limit=1e6,
)
frames_gauss_fd3_full = propagate_linear_frames_with_source(
    preparedFD3,
    Nt;
    initialPast=initial,
    initialPresent=initial,
    store_every=store_every,
    blowup_limit=1e6,
)
ux_gauss = component_frames(frames_gauss_full, 1)
uz_gauss = component_frames(frames_gauss_full, 2)
ux_gauss_fd3 = component_frames(frames_gauss_fd3_full, 1)
uz_gauss_fd3 = component_frames(frames_gauss_fd3_full, 2)
@show wavefield_snapshot_report(ux_gauss)[end] wavefield_snapshot_report(uz_gauss)[end]
@show wavefield_snapshot_report(ux_gauss_fd3)[end] wavefield_snapshot_report(uz_gauss_fd3)[end]
@show frame_difference_report(uz_gauss, uz_gauss_fd3)[end]

# Moment-source test for Γ: M11 only.
timeSignal = source_time_samples(Nt, dt, preparedElastic.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceFull = point_source_full(preparedElastic, sourcePoint, timeSignal; iForceField=1, amplitude=source_amplitude)
source_peak_index = argmax(abs.(timeSignal))
source_peak_time = (source_peak_index - 1) * dt
source_rhs = source_rhs_diagnostics(preparedElastic, sourceFull; it=min(source_peak_index, Nt))
@show maximum(abs, timeSignal) source_peak_index source_peak_time maximum(abs, sourceFull)
@show source_rhs.b_max source_rhs.b_norm source_rhs.b_argmax

frames_elastic_full = propagate_linear_frames_with_source(
    preparedElastic,
    Nt;
    sourceFull=sourceFull,
    store_every=store_every,
    blowup_limit=1e6,
)
ux_frames = component_frames(frames_elastic_full, 1)
uz_frames = component_frames(frames_elastic_full, 2)

# Direct vertical body-force comparison, fx=0 and fz=1.
timeSignal_force = source_time_samples(Nt, dt, preparedElastic_force.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceFull_fz = direct_force_source_full(preparedElastic_force, sourcePoint, timeSignal_force; fx=0.0, fz=1.0, amplitude=source_amplitude)
sourceFull_fz_fd3 = direct_force_source_full(preparedFD3, sourcePoint, timeSignal_force; fx=0.0, fz=1.0, amplitude=source_amplitude)
source_rhs_force = source_rhs_diagnostics(preparedElastic_force, sourceFull_fz; it=min(argmax(abs.(timeSignal_force)), Nt))
@show source_rhs_force.b_max source_rhs_force.b_norm source_rhs_force.b_argmax maximum(abs, sourceFull_fz)

frames_force_full = propagate_linear_frames_with_source(
    preparedElastic_force,
    Nt;
    sourceFull=sourceFull_fz,
    store_every=store_every,
    blowup_limit=1e6,
)
frames_force_fd3_full = propagate_linear_frames_with_source(
    preparedFD3,
    Nt;
    sourceFull=sourceFull_fz_fd3,
    store_every=store_every,
    blowup_limit=1e6,
)
ux_force = component_frames(frames_force_full, 1)
uz_force = component_frames(frames_force_full, 2)
ux_force_fd3 = component_frames(frames_force_fd3_full, 1)
uz_force_fd3 = component_frames(frames_force_fd3_full, 2)

ux_report = wavefield_snapshot_report(ux_frames)
uz_report = wavefield_snapshot_report(uz_frames)
uz_force_report = wavefield_snapshot_report(uz_force)
uz_force_fd3_report = wavefield_snapshot_report(uz_force_fd3)
@show length(frames_elastic_full) ux_report[1] ux_report[end]
@show uz_report[1] uz_report[end]
@show uz_force_report[1] uz_force_report[end]
@show uz_force_fd3_report[1] uz_force_fd3_report[end]
@show frame_difference_report(uz_force, uz_force_fd3)[end]


In [ ]:
display(plot_wave_snapshots(ux_gauss; sourcePoint=sourcePoint, title="Marmousi elastic OPT Gaussian ux"))
display(plot_wave_snapshots(uz_gauss; sourcePoint=sourcePoint, title="Marmousi elastic OPT Gaussian uz"))
display(plot_wave_snapshots(ux_gauss_fd3; sourcePoint=sourcePoint, title="Marmousi elastic FD3 Gaussian ux"))
display(plot_wave_snapshots(uz_gauss_fd3; sourcePoint=sourcePoint, title="Marmousi elastic FD3 Gaussian uz"))


In [ ]:
display(plot_wave_snapshots(uz_frames; sourcePoint=sourcePoint, title="Marmousi elastic OPT moment M11 uz"))
display(plot_wave_snapshots(ux_force; sourcePoint=sourcePoint, title="Marmousi elastic OPT fz source ux"))
display(plot_wave_snapshots(uz_force; sourcePoint=sourcePoint, title="Marmousi elastic OPT fz source uz"))
display(plot_wave_snapshots(ux_force_fd3; sourcePoint=sourcePoint, title="Marmousi elastic FD3 fz source ux"))
display(plot_wave_snapshots(uz_force_fd3; sourcePoint=sourcePoint, title="Marmousi elastic FD3 fz source uz"))


## 6. Local Stencil Inspection


In [ ]:
st_elastic_eq1_u1 = operator_stencil_at_point(elasticOPT.numOps, sourcePoint; which=:left, iExpr=1, iField=1)
st_elastic_eq2_u2 = operator_stencil_at_point(elasticOPT.numOps, sourcePoint; which=:left, iExpr=2, iField=2)

stencil_time_summary(st_elastic_eq1_u1), stencil_time_summary(st_elastic_eq2_u2)


## 7. Larger Native-Crop Marmousi Double Couple Videos

This keeps the previous debug cells intact and builds a larger Marmousi crop with `downsample_step_dc = 1`. OPT uses the moment-tensor RHS (`M11,M12,M21,M22`); FD3 uses the equivalent centered finite-difference divergence of the same point moment.


In [ ]:
shape_dc = (121, 121)
downsample_step_dc = 1
rho_raw_dc = downsample_center_crop(marmousi.ρ, shape_dc; step=downsample_step_dc)
vp_raw_dc = downsample_center_crop(marmousi.Vpv, shape_dc; step=downsample_step_dc)
vs_raw_dc = downsample_center_crop(marmousi.Vsv, shape_dc; step=downsample_step_dc)

rho_dc, lambda_dc, mu_dc, vp_dc, vs_dc = elastic_lame_from_rho_vp_vs(rho_raw_dc, vp_raw_dc, vs_raw_dc)

vs_floor_dc = 500.0
if vs_floor_dc > 0
    vs_dc = max.(vs_dc, vs_floor_dc)
    mu_dc = rho_dc .* vs_dc.^2
    lambda_dc = rho_dc .* vp_dc.^2 .- 2 .* mu_dc
end

dx_dc = 1.0e3 * downsample_step_dc
cfl_dc = 0.18
cfl_info_dc = cfl_diagnostics(vp_dc, (dx_dc, dx_dc, 1.0); cfl_safety=cfl_dc)
dt_dc = cfl_info_dc.suggested_dt_2D
delta_dc_mks = (dx_dc, dx_dc, dt_dc)

rho_nd_dc = ones(Float64, shape_dc)
lambda_nd_dc = (lambda_dc ./ rho_dc) .* (dt_dc / dx_dc)^2
mu_nd_dc = (mu_dc ./ rho_dc) .* (dt_dc / dx_dc)^2
models_elastic_dc = [rho_nd_dc, lambda_nd_dc, mu_nd_dc]
delta_dc = (1.0, 1.0, 1.0)

sourcePoint_dc = CartesianIndex(cld(shape_dc[1], 2), cld(shape_dc[2], 2))
Nt_dc = 220
store_every_dc = 4
total_time_dc = Nt_dc * dt_dc
source_f0_dc = min(max(cfl_info_dc.suggested_f0, 0.08), 1 / (8dt_dc))
source_t0_dc = min(12dt_dc, 0.25total_time_dc)
source_amplitude_dc = 1.0

@show size(rho_dc) extrema(vp_dc) extrema(vs_dc)
@show delta_dc_mks extrema(lambda_nd_dc) extrema(mu_nd_dc)
@show sourcePoint_dc Nt_dc store_every_dc total_time_dc source_f0_dc source_t0_dc
@show maximum(vp_dc) * total_time_dc / dx_dc maximum(vs_dc) * total_time_dc / dx_dc


In [ ]:
elasticOPT_dc = build_opt_prepared(
    "2DsismoTimeIsoHetero",
    models_elastic_dc,
    delta_dc;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="Marmousi_native_crop_double_couple_OPT",
)
preparedOPT_dc = elasticOPT_dc.prepared

preparedFD3_dc = prepare_fd2d_elastic_pointwise_baseline(
    rho_nd_dc,
    lambda_nd_dc,
    mu_nd_dc,
    delta_dc;
    spatial_order=3,
    source_scale=1.0,
)

@show preparedOPT_dc.spaceShape preparedOPT_dc.NpointsSpace preparedOPT_dc.NField preparedOPT_dc.NForceField
@show nnz(preparedOPT_dc.A_unknown) nnz(preparedOPT_dc.L_known) nnz(preparedOPT_dc.R_force)
@show nnz(preparedFD3_dc.A_unknown) nnz(preparedFD3_dc.L_known) nnz(preparedFD3_dc.R_force)
@show implicit_matrix_report(preparedOPT_dc)
@show implicit_matrix_report(preparedFD3_dc)


In [ ]:
timeSignal_dc = source_time_samples(Nt_dc, dt_dc, preparedOPT_dc.timePointsUsedForOneStep; t0=source_t0_dc, f0=source_f0_dc)

# Double couple convention for this debug cell: M11 + M22 = 0 and M12 = M21.
M11_dc = 1.0
M22_dc = -1.0
M12_dc = 1.0
M21_dc = M12_dc

sourceFull_dc_opt = double_couple_moment_source_full(
    preparedOPT_dc,
    sourcePoint_dc,
    timeSignal_dc;
    M11=M11_dc,
    M22=M22_dc,
    M12=M12_dc,
    M21=M21_dc,
    amplitude=source_amplitude_dc,
)
sourceFull_dc_fd3 = fd_double_couple_force_source_full(
    preparedFD3_dc,
    sourcePoint_dc,
    timeSignal_dc;
    M11=M11_dc,
    M22=M22_dc,
    M12=M12_dc,
    M21=M21_dc,
    dx=delta_dc[1],
    dz=delta_dc[2],
    amplitude=source_amplitude_dc,
    sign=1.0,
)

source_peak_index_dc = argmax(abs.(timeSignal_dc))
@show maximum(abs, timeSignal_dc) source_peak_index_dc (source_peak_index_dc - 1) * dt_dc
@show source_rhs_diagnostics(preparedOPT_dc, sourceFull_dc_opt; it=min(source_peak_index_dc, Nt_dc)).b_max
@show source_rhs_diagnostics(preparedFD3_dc, sourceFull_dc_fd3; it=min(source_peak_index_dc, Nt_dc)).b_max

frames_dc_opt_full = propagate_linear_frames_with_source(
    preparedOPT_dc,
    Nt_dc;
    sourceFull=sourceFull_dc_opt,
    store_every=store_every_dc,
    blowup_limit=1e8,
)
frames_dc_fd3_full = propagate_linear_frames_with_source(
    preparedFD3_dc,
    Nt_dc;
    sourceFull=sourceFull_dc_fd3,
    store_every=store_every_dc,
    blowup_limit=1e8,
)

ux_dc_opt = component_frames(frames_dc_opt_full, 1)
uz_dc_opt = component_frames(frames_dc_opt_full, 2)
ux_dc_fd3 = component_frames(frames_dc_fd3_full, 1)
uz_dc_fd3 = component_frames(frames_dc_fd3_full, 2)

@show wavefield_snapshot_report(uz_dc_opt)[end]
@show wavefield_snapshot_report(uz_dc_fd3)[end]
@show frame_difference_report(uz_dc_opt, uz_dc_fd3)[end]


In [ ]:
display(plot_wave_snapshots(uz_dc_opt; sourcePoint=sourcePoint_dc, title="Marmousi native crop OPT double couple uz"))
display(plot_wave_snapshots(uz_dc_fd3; sourcePoint=sourcePoint_dc, title="Marmousi native crop FD3 double couple uz"))


In [ ]:
video_dir = joinpath(@__DIR__, "tmp", "videos")
mkpath(video_dir)
opt_dc_video = joinpath(video_dir, "marmousi_native_double_couple_OPT_uz.mp4")
fd3_dc_video = joinpath(video_dir, "marmousi_native_double_couple_FD3_uz.mp4")

common_clim_dc = maximum((maximum(maximum(abs, f) for f in uz_dc_opt), maximum(maximum(abs, f) for f in uz_dc_fd3)))
record_wave_component_video(
    uz_dc_opt;
    videoFile=opt_dc_video,
    sourcePoint=sourcePoint_dc,
    background=vp_dc,
    framerate=20,
    title="OPT double couple uz",
    clim=common_clim_dc,
)
record_wave_component_video(
    uz_dc_fd3;
    videoFile=fd3_dc_video,
    sourcePoint=sourcePoint_dc,
    background=vp_dc,
    framerate=20,
    title="FD3 double couple uz",
    clim=common_clim_dc,
)

@show opt_dc_video fd3_dc_video
